# Notebook 9: Final Prescriptive Validation (Baseline-to-Target)

## 1. Introduction
This is the final validation notebook for the Prescriptive Analytics engine. The goal is to verify that the **Delta Optimizer** reacts logically to different production states.

We are testing the system's ability to:
1. **Detect State**: Correctly identify if the current baseline is Overproducing or Underproducing.
2. **Directional Adjustment**: 
    - In **Underproduction**, it must suggest *increasing* resources.
    - In **Overproduction**, it must suggest *reducing* resources.
    - In **Balanced** state, it must suggest *no change*.
3. **Accuracy**: Get as close as possible to the `planned_qty` using the smallest possible change from the baseline.

In [1]:
import pandas as pd
import numpy as np
import joblib
import itertools

# 1. Load Model
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

# 2. Integrate Optimizer Logic
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {'min': 8, 'max': 32, 'step': 1},
    'working_hours': {'min': 200.0, 'max': 1600.0, 'step': 8.0},
    'machine_count': {'min': 2, 'max': 10, 'step': 1}
}

def build_prescriptive_features(raw_inputs):
    features = raw_inputs.copy()
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers
    defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
    for col, value in defaults.items():
        if col not in features: features[col] = value
    return pd.DataFrame([features])

class DeltaOptimizer:
    def __init__(self, model, constraints):
        self.model = model
        self.constraints = constraints
        self.w_gap = 1.0
        self.w_delta = 0.1
    
    def generate_search_space(self):
        workers = range(self.constraints['no_of_workers']['min'], self.constraints['no_of_workers']['max'] + 1, self.constraints['no_of_workers']['step'])
        hours = np.arange(self.constraints['working_hours']['min'], self.constraints['working_hours']['max'] + 0.1, self.constraints['working_hours']['step'])
        machines = range(self.constraints['machine_count']['min'], self.constraints['machine_count']['max'] + 1, self.constraints['machine_count']['step'])
        return list(itertools.product(workers, hours, machines))
    
    def calculate_resource_cost(self, df):
        return ((df['no_of_workers']/32) + (df['working_hours']/1500) + (df['machine_count']/7)) / 3
    
    def optimize(self, current_resources, planned_qty, context_vars):
        full_baseline_inputs = {**current_resources, **context_vars}
        baseline_feat = build_prescriptive_features(full_baseline_inputs)
        baseline_pred = self.model.predict(baseline_feat)[0]
        
        gap = baseline_pred - planned_qty
        if abs(gap) <= 2:
            return "Balanced", None
        
        state = "Overproduction" if gap > 2 else "Underproduction"
        search_space = self.generate_search_space()
        df_scenarios = pd.DataFrame(search_space, columns=['no_of_workers', 'working_hours', 'machine_count'])
        
        if state == "Overproduction":
            df_scenarios = df_scenarios[(df_scenarios['no_of_workers'] <= current_resources['no_of_workers']) & (df_scenarios['working_hours'] <= current_resources['working_hours']) & (df_scenarios['machine_count'] <= current_resources['machine_count'])].copy()
        elif state == "Underproduction":
            df_scenarios = df_scenarios[(df_scenarios['no_of_workers'] >= current_resources['no_of_workers']) & (df_scenarios['working_hours'] >= current_resources['working_hours']) & (df_scenarios['machine_count'] >= current_resources['machine_count'])].copy()
        
        for col, val in context_vars.items(): df_scenarios[col] = val
        df_scenarios['machine_worker_ratio'] = df_scenarios['machine_count'] / df_scenarios['no_of_workers']
        df_scenarios['avg_hours_per_worker'] = df_scenarios['working_hours'] / df_scenarios['no_of_workers']
        defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
        for col, value in defaults.items():
            if col not in df_scenarios.columns: df_scenarios[col] = value
        
        predictions = self.model.predict(df_scenarios)
        df_scenarios['predicted_qty'] = predictions
        df_scenarios['gap'] = abs(planned_qty - df_scenarios['predicted_qty'])
        df_scenarios['cost'] = self.calculate_resource_cost(df_scenarios)
        df_scenarios['score'] = (self.w_gap * df_scenarios['gap']) + (self.w_delta * df_scenarios['cost'])
        return state, df_scenarios.sort_values('score').head(1)

optimizer = DeltaOptimizer(model, RESOURCE_CONSTRAINTS)
print("Validation environment ready.")

Validation environment ready.


## 2. Stress Testing the Prescriptive Logic

We will now run the optimizer against three specific scenarios to prove that the "Directional Search" is working.

In [3]:
context = {
    'item_name': 'DIYANA - L - BROWN',
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Gents',
    'size_id': 10,
    'grade': 'A',
    'order_category': 'Others'
}

current_res = {'no_of_workers': 15, 'working_hours': 623.96, 'machine_count': 4}

test_scenarios = {
    'Underproduction Case': 70,   # Target is much higher than baseline (~35)
    'Overproduction Case': 15,    # Target is much lower than baseline
    'Balanced Case': 35          # Target is almost equal to baseline
}

results = []

for name, target in test_scenarios.items():
    state, recs = optimizer.optimize(current_res, target, context)
    
    if recs is not None:
        best = recs.iloc[0]
        res_summary = {
            'Scenario': name,
            'Target': target,
            'State': state,
            'Rec_Workers': best['no_of_workers'],
            'Rec_Hours': best['working_hours'],
            'Rec_Machines': best['machine_count'],
            'Pred_Qty': best['predicted_qty'],
            'Gap': best['gap']
        }
    else:
        res_summary = {
            'Scenario': name, 'Target': target, 'State': state, 
            'Rec_Workers': current_res['no_of_workers'], 
            'Rec_Hours': current_res['working_hours'], 
            'Rec_Machines': current_res['machine_count'], 
            'Pred_Qty': 'N/A', 'Gap': 'N/A'
        }
    results.append(res_summary)

df_validation = pd.DataFrame(results)
display(df_validation)

,Scenario,Target,State,Rec_Workers,Rec_Hours,Rec_Machines,Pred_Qty,Gap
0,Underproduction Case,70,Underproduction,19,1112.00,6,70.003384,0.003384
1,Overproduction Case,15,Overproduction,14,208.00,2,15.005018,0.005018
2,Balanced Case,35,Balanced,15,623.96,4,N/A,N/A


## 3. Final Verdict

To confirm the project is a success, verify the following in the table:
1. **Underproduction**: Are Workers/Hours/Machines **$\ge$** 15/623/4?
2. **Overproduction**: Are Workers/Hours/Machines **$\le$** 15/623/4?
3. **Balanced**: Does it say "Balanced" and keep resources the same?

If these three conditions are met, the Prescriptive Analytics engine is logically consistent, directionally correct, and ready for the Streamlit application.